In [1]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import re
import psycopg2
from functions import read_db_credentials, connect_to_db, load_json_data_from_db_as_json, save_df_to_db, data_from_data_sink

In [2]:
def save_df_to_db(df, table_name):
    creds = read_db_credentials()
    conn = connect_to_db(creds)
    cursor = conn.cursor()

    df = df.where(pd.notnull(df), None)  # NaN → None

    columns = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(df.columns))

    # dynamisch UPDATE-Anweisung generieren
    update_assignments = ', '.join([
        f"{col} = EXCLUDED.{col}" for col in df.columns if col != "user_number"
    ])

    insert_query = f"""
        INSERT INTO {table_name} ({columns})
        VALUES ({placeholders})
        ON CONFLICT (user_number) DO UPDATE
        SET {update_assignments}
    """

    for _, row in df.iterrows():
        cursor.execute(insert_query, row.tolist())

    conn.commit()
    cursor.close()
    conn.close()



In [3]:
with open("data/cur_user_selected.txt", "r", encoding="utf-8") as f:
    user = int(f.read().strip())

# with open("linkedin_profile Kopie.json") as f:
#     linkedin_data = json.load(f)
# def load_json_data_from_db_as_json(user, source ):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)

#     query = f"SELECT raw_json FROM fact_raw_data WHERE data_source = '{source}' AND user_number = {user} ;"
    
#     cursor = conn.cursor()
#     cursor.execute(query)
#     result = cursor.fetchone()
#     conn.close()
#     #return (result)
#     # # Parsen der JSON-Inhalte aus der 'data'-Spalte
#     parsed_data = result[0] if result else {}


#     # # Rückgabe als JSON-String (optional indent für Lesbarkeit)
#     return parsed_data


linkedin_data = load_json_data_from_db_as_json(user, "linkedin")
print(linkedin_data)

{'name': 'Kontakt', 'title': 'Top-Kenntnisse', 'location': None, 'contact': {'email': 'dave.emmert2304@icloud.com', 'linkedin': 'https://www.linkedin.com/in/david-', 'portfolio': ''}, 'skills': [], 'certifications': [], 'summary': '', 'experience': [], 'total_experience_years': 0}


In [4]:
# Top-Level Keys anzeigen
data = linkedin_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


"name": str
"title": str
"location": NoneType
"contact": 
  "email": str
  "linkedin": str
  "portfolio": str
"skills": 
  [
    <empty>
  ]
"certifications": 
  [
    <empty>
  ]
"summary": str
"experience": 
  [
    <empty>
  ]
"total_experience_years": int


In [5]:
def categorize_experience(years) -> str:
    """
    Categorizes work experience based on total years.

    Categories:
    - Junior: < 2 years
    - Senior: 2-6 years
    - Expert: > 6 years

    :param profile: Dictionary with key "total_experience_years"
    :return: Category as a string
    """
    
    if years < 2:
        return "Junior"
    elif 3 <= years <= 6:
        return "Senior"
    else:
        return "Expert"


# Experience

In [6]:
exp_years = linkedin_data.get("total_experience_years", 0.0)
print(exp_years)
exp_type = categorize_experience(exp_years)
print(exp_type)


0
Junior


# SKILLS

In [7]:
skills = [
    "Python", "R", "SQL", "Excel", "Power BI", "Tableau", "Pandas", "NumPy",
    "Statistics", "Data Analysis", "Data Visualization", "Machine Learning",
    "Deep Learning", "Scikit-learn", "TensorFlow", "PyTorch", "Keras",
    "Google Analytics", "Data Engineering", "ETL", "BigQuery", "Apache Spark",
    "Databricks", "Snowflake", "Looker", "Data Mining", "A/B Testing",
    "Java", "JavaScript", "TypeScript", "HTML", "CSS", "React", "Angular",
    "Vue.js", "Node.js", "PHP", "C++", "C#", ".NET", "Spring Boot", "Django",
    "Flask", "Ruby on Rails", "FastAPI", "Next.js", "Express.js", "GraphQL",
    "REST APIs", "Git", "GitHub", "GitLab", "Bitbucket", "Jest", "Cypress",
    "Selenium", "Playwright", "Webpack", "Babel", "NPM", "Yarn", "Linux",
    "Shell Scripting", "Bash", "Software Architecture", "Microservices",
    "AWS", "Azure", "Google Cloud Platform", "GCP", "Docker", "Kubernetes",
    "Terraform", "Ansible", "CI/CD", "Jenkins", "GitHub Actions", "CircleCI",
    "Travis CI", "CloudFormation", "Prometheus", "Grafana", "New Relic",
    "ELK Stack", "Logstash", "Kibana", "Helm", "Istio", "Serverless", "Vagrant",
    "MySQL", "PostgreSQL", "MongoDB", "Redis", "SQLite", "Oracle", "MariaDB",
    "Firebase", "Cassandra", "DynamoDB", "Couchbase", "Elasticsearch", "InfluxDB",
    "Agile", "Scrum", "Kanban", "Jira", "Confluence", "Trello", "Asana",
    "Project Management", "Product Management", "Stakeholder Management",
    "Requirement Analysis", "Waterfall", "PRINCE2", "SAFe", "Lean",
    "Business Analysis", "Process Optimization", "Digital Transformation",
    "Stakeholder Communication", "Risk Management", "Business Intelligence",
    "Decision Making", "OKRs", "KPIs", "Roadmapping", "User Stories",
    "SEO", "SEM", "Google Ads", "Facebook Ads", "Instagram Ads", "LinkedIn Ads",
    "Marketing Automation", "HubSpot", "Salesforce", "Content Marketing",
    "Email Marketing", "Campaign Management", "Social Media Marketing",
    "Influencer Marketing", "Lead Generation", "CRM", "Brand Strategy",
    "UI Design", "UX Design", "Figma", "Sketch", "Adobe XD", "Adobe Photoshop",
    "Adobe Illustrator", "InVision", "Wireframing", "Prototyping",
    "Design Thinking", "Interaction Design", "User Research", "Motion Design",
    "Typography", "Responsive Design", "Branding", "Accessibility",
    "Financial Modeling", "Budgeting", "Forecasting", "QuickBooks", "Xero",
    "SAP", "IFRS", "GAAP", "Compliance", "Tax Law", "Legal Research",
    "Contract Management", "Due Diligence", "Audit", "Cost Analysis",
    "Communication", "Leadership", "Teamwork", "Problem Solving",
    "Time Management", "Critical Thinking", "Adaptability", "Empathy",
    "Conflict Resolution", "Emotional Intelligence", "Creativity",
    "Presentation Skills", "Negotiation", "Decision Making", "Mentoring",
    "Computer Vision", "Natural Language Processing", "NLP", "OCR", "Speech Recognition",
    "AR/VR", "Blockchain", "Web3", "Smart Contracts", "IoT", "Edge Computing",
    "Cybersecurity", "Penetration Testing", "Network Security", "Ethical Hacking",
    "SaaS", "PaaS", "IaaS", "API Design", "WebSockets", "OAuth2", "SAML",
    "English", "German", "Spanish", "French", "Hindi", "Mandarin", "Arabic", "Portuguese", "Russian","Data"
]

# Regex vorbereiten: Wortgrenzen, escape für Sonderzeichen
def to_regex(skill):
    return r'\b' + re.escape(skill) + r'\b'
# Beispielhafte Bewertungsskala:
# 5 = sehr wertvoll (Schlüsseltechnologien, gefragte Skills)
# 4 = hoch (oft gefordert, zentrale Tools)
# 3 = mittel (relevant, aber eher unterstützend)
# 2 = gering (spezifisch oder ergänzend)
# 1 = nice to have (nischig oder optional)

# Bewertungsliste (vereinfachte Einteilung nach Bedeutung/Verbreitung)
high_value = {
    4: ["Python", "SQL", "Machine Learning", "AWS", "Azure", "JavaScript", "React", "Docker", "Git",  "Communication", "Project Management"],
    3: ["Tableau", "Power BI", "Natural Language Processing","TensorFlow", "NLP","Pandas", "NumPy", "Java", "Node.js", "GitHub", "Jenkins", "Excel","Kubernetes", "Google Cloud Platform", "Agile", "Scrum", "Leadership", "Data Analysis", "Problem Solving", "HTML", "CSS", "Data Engineering", "Django", "C#", "MySQL", "PostgreSQL", "MongoDB"],
}

management_keywords = [
        "Project Management", "Product Management", "Stakeholder Management", "Leadership", "Scrum", "Agile",
        "Kanban", "Risk Management", "Roadmapping", "OKRs", "KPIs", "Jira", "Confluence", "Trello", "Asana",
        "SAFe", "Waterfall", "PRINCE2", "Lean", "Business Analysis", "Digital Transformation"
    ]
it_keywords = [
            "Python", "R", "SQL", "Java", "JavaScript", "TypeScript", "HTML", "CSS", "PHP", "C++", "C#", ".NET",
            "Spring Boot", "Django", "Flask", "Ruby on Rails", "FastAPI", "Next.js", "Express.js", "GraphQL",
            "REST APIs", "Git", "GitHub", "GitLab", "Bitbucket", "Jest", "Cypress", "Selenium", "Playwright",
            "Webpack", "Babel", "NPM", "Yarn", "Linux", "Shell Scripting", "Bash", "Software Architecture",
            "Microservices", "AWS", "Azure", "Google Cloud Platform", "GCP", "Docker", "Kubernetes",
            "Terraform", "Ansible", "CI/CD", "Jenkins", "GitHub Actions", "CircleCI", "Travis CI", "CloudFormation",
            "Prometheus", "Grafana", "New Relic", "ELK Stack", "Logstash", "Kibana", "Helm", "Istio", "Serverless",
            "Vagrant", "MySQL", "PostgreSQL", "MongoDB", "Redis", "SQLite", "Oracle", "MariaDB", "Firebase","Data"
            "Cassandra", "DynamoDB", "Couchbase", "Elasticsearch", "InfluxDB", "Looker", "Snowflake", "Databricks",
            "BigQuery", "ETL", "Apache Spark", "Tableau", "Power BI", "Google Analytics", "Data Engineering",
            "Data Mining", "A/B Testing", "Scikit-learn", "TensorFlow", "PyTorch", "Keras", "Statistics", "Pandas",
            "NumPy", "Computer Vision", "Natural Language Processing", "NLP", "OCR", "Speech Recognition",
            "AR/VR", "Blockchain", "Web3", "Smart Contracts", "IoT", "Edge Computing", "Cybersecurity",
            "Penetration Testing", "Network Security", "Ethical Hacking", "SaaS", "PaaS", "IaaS", "API Design",
            "WebSockets", "OAuth2", "SAML", "Figma", "Sketch", "Adobe XD", "Adobe Photoshop", "Adobe Illustrator",
            "InVision"
        ]
communication_keywords = [
        "Communication", "Presentation Skills", "Negotiation", "Stakeholder Communication", "Teamwork", "Mentoring",
        "Empathy", "Conflict Resolution", "Emotional Intelligence", "English", "German", "Spanish", "French",
        "Hindi", "Mandarin", "Arabic", "Portuguese", "Russian"
    ]
problem_solving_keywords = [
        "Problem Solving", "Decision Making", "Critical Thinking", "Adaptability", "Creativity",
        "Process Optimization", "Data Analysis", "Data Visualization", "Business Intelligence",
        "Requirement Analysis", "User Research", "Design Thinking", "Interaction Design", "Wireframing",
        "Prototyping", "Motion Design", "Typography", "Responsive Design", "Branding", "Accessibility"
    ]
def assign_category(skill):
    if skill in management_keywords:
        return "Management"
    elif skill in it_keywords:
        return "IT"
    elif skill in communication_keywords:
        return "Communication"
    elif skill in problem_solving_keywords:
        return "Problem Solving"
    else:
        return "Others"



# Skill-Rating berechnen
def rate_skill(skill):
    for rating, keywords in high_value.items():
        if skill in keywords:
            return rating
    return 2 if skill in df_skills["skill"].values else 1

# DataFrame erstellen
df_skills = pd.DataFrame({
    "skill": skills,
    "regex": [to_regex(skill) for skill in skills]
})
df_skills["rating"] = df_skills["skill"].apply(rate_skill)
df_skills["category"] = df_skills["skill"].apply(assign_category)

# Funktion zum Matchen von Skills anhand Regex mit Rückgabe von Rating und Kategorie
def match_skills_with_metadata(skills_to_match, skill_df):
    matched = []
    for input_skill in skills_to_match:
        found = False
        for _, row in skill_df.iterrows():
            if re.search(row["regex"], input_skill, re.IGNORECASE):
                matched.append({
                    "input": input_skill,
                    "matched_skill": row["skill"],
                    "rating": row["rating"],
                    "category": row["category"]
                })
                found = True
                break
        if not found:
            matched.append({
                "input": input_skill,
                "matched_skill": None,
                "rating": None,
                "category": "Not Found"
            })
    matched = pd.DataFrame(matched)
    matched["length_score"] = (matched["input"].str.len() - 13).abs()
    matched = matched.sort_values(by=["rating", "length_score"], ascending=[False, True]).reset_index()
    matched = matched.drop(columns=["length_score"])
    return matched


In [8]:
skills = linkedin_data.get("skills")

if not skills:  # None, [] oder andere leere Werte
    skills = ["not maintained", "not maintained", "not maintained"]
print(skills)
print(df_skills)
skills_processed = match_skills_with_metadata(skills, df_skills)
print(skills_processed)
top_skill_1 = skills_processed["input"][0]
top_skill_2 = skills_processed["input"][1]
top_skill_3 = skills_processed["input"][2]

category_counts = skills_processed["category"].value_counts(normalize=True) * 100

# In DataFrame umwandeln
df_category_distribution = category_counts.reset_index()
df_category_distribution.columns = ["category", "relative_frequency_percent"]

skill_cat_1 = f'{df_category_distribution["category"][0]} ({df_category_distribution["relative_frequency_percent"][0]}%)'
if len(df_category_distribution) > 1:
    skill_cat_2 = f'{df_category_distribution["category"][1]} ({df_category_distribution["relative_frequency_percent"][1]}%)'
else:
    skill_cat_2 = "NA"

['not maintained', 'not maintained', 'not maintained']
          skill           regex  rating       category
0        Python      \bPython\b       4             IT
1             R           \bR\b       2             IT
2           SQL         \bSQL\b       4             IT
3         Excel       \bExcel\b       3         Others
4      Power BI   \bPower\ BI\b       3             IT
..          ...             ...     ...            ...
221    Mandarin    \bMandarin\b       2  Communication
222      Arabic      \bArabic\b       2  Communication
223  Portuguese  \bPortuguese\b       2  Communication
224     Russian     \bRussian\b       2  Communication
225        Data        \bData\b       2         Others

[226 rows x 4 columns]
   index           input matched_skill rating   category
0      0  not maintained          None   None  Not Found
1      1  not maintained          None   None  Not Found
2      2  not maintained          None   None  Not Found


# SKILL DIVERSITY INDEX

In [9]:
valid_categories = skills_processed['category'].dropna().unique()
num_categories = len(valid_categories)

# Score berechnen
score = num_categories / 5  # Es gibt 5 mögliche Kategorien laut Diagramm

# Klassifikation anhand des Scores
if score > 0.6:
    classification = "Well Rounded"
elif score > 0.4:
    classification = "Specialised"
else:
    classification = "Focused"

skill_diversity_index = score
skill_diversity_cat = classification

In [10]:
if len(linkedin_data.get("experience"))>0:
    if (linkedin_data.get("experience")[0]["end_date"] == "Present"):
        date = f'since {linkedin_data.get("experience")[0]["start_date"]}'
    else:
        date = f'from  {linkedin_data.get("experience")[0]["start_date"]} until {linkedin_data.get("experience")[0]["end_date"]}'

    last_postion = f'{linkedin_data.get("experience")[0]["position"]} at {linkedin_data.get("experience")[0]["company"]} in {linkedin_data.get("experience")[0]["location"]} {date}' 
    last_postion
else: 
    last_postion = "not found"

In [11]:

activity_times = data_from_data_sink(f"WITH id AS ( SELECT DISTINCT at_id FROM dim_health WHERE user_number = {user}) SELECT * FROM dim_activity_times WHERE at_id IN (SELECT at_id FROM id);")


/Users/dave/Desktop/BIPM/WHO_AM_I/sports-buddies/sports-buddies/functions.py:84: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [12]:
import pandas as pd

def classify_working_type(df: pd.DataFrame) -> str:
    """
    Klassifiziert das Aktivitätsmuster einer Person basierend auf dem Zeitanteil
    früher, später und Wochenendaktivität.

    Voraussetzungen:
    - df enthält Spalten: 'weekday' (0=Montag, ..., 6=Sonntag), 'day_part' (Early, Late, Typical)

    Returns:
    - Kategorie als String: Early Bird, Midnight, Weekend, Typical
    """
    total = len(df)
    if total == 0:
        return "No Data"

    total = len(df)

    # Definierte Tageszeiten-Kategorien
    early_parts = ["Early Morning\n(4–8)", "Morning\n(8–12) "]
    late_parts = ["Evening\n(20–24)", "Night\n(0–4)"]
    weekend = ["Saturday", "Sunday"]

    early_pct = df["day_part"].isin(early_parts).sum() / total * 100
    late_pct = df["day_part"].isin(late_parts).sum() / total * 100
    weekend_pct = df["weekday"].isin(weekend ).sum() / total * 100

    if early_pct > 40:
        return "Early Bird"
    elif late_pct > 30:
        return "Midnight"
    elif weekend_pct > 50:
        return "Weekend"
    else:
        return "Typical"


working_type = classify_working_type(activity_times)

In [14]:

res = pd.DataFrame([{
    "user_number": user,
    "exp_years": exp_years,
    "exp_type": exp_type,
    "top_skill_1": top_skill_1,
    "top_skill_2": top_skill_2,
    "top_skill_3": top_skill_3,
    "skill_div_cat": skill_diversity_cat,
    "skill_div_index": skill_diversity_index,
    "skill_cat_1": skill_cat_1,
    "skill_cat_2": skill_cat_2 ,
    "cur_job": last_postion,
    "working_type": working_type
}])
print(res)


save_df_to_db(res, "dim_linkedin")

   user_number  exp_years exp_type     top_skill_1     top_skill_2  \
0           10          0   Junior  not maintained  not maintained   

      top_skill_3 skill_div_cat  skill_div_index         skill_cat_1  \
0  not maintained       Focused              0.2  Not Found (100.0%)   

  skill_cat_2    cur_job working_type  
0          NA  not found     Midnight  
